In [ ]:
import importlib
for pkg in ["numpy", "matplotlib"]:
    importlib.import_module(pkg)
print("All libraries are installed.")

In [ ]:
from bloch import *
from visualizations import *
from helpers import *


# Lab 4: Gradients and Spatial Encoding

Howdy! Welcome to dry lab number four. Last time, you learned how to tip over (excite) an _ensemble_ of spins with an RF pulse, and you even turned on a *gradient* so as to excite only a thin slice. In lab 1, we had a very brief introduction to fourier transforms and a convenient little thing called ***k-space***. Further, you've learned in class that MRI data _lives_ in k-space, and that the reconstructed image is never more than a Fourier transform away.

In this lab, we're going to use the Bloch simulator to *collect a signal* while a gradient is on, make a connection between that signal and the object's k-space, and then Fourier-transform our way back to an image.

## Part 1: Starting with $1\text{D}$

Recall that in Lab 3, we applied a gradient in $\hat z$ during the RF pulse to excite only a slice. This was, in some sense, spatial encoding: we made sure that the image came from only a very narrow band of $z$ coordinates. In doing so we reduced a $3\text{D}$ imaging problem to a $2\text{D}$ one. I think you could try to use a very similar strategy to recover a brightness / intensity _one voxel at a time_. Explain how, and then explain why this is probably not used in practice. 

<span style="color:cyan">
Your answer here.
</span>

We're going to learn a more efficient method for spatial encoding of an image now. This is usually called _fourier transform_ or _spin-warp_ imaging. At a high level, we will use gradients to apply __spatially dependent and time varying phase__ to the image; and since the receive coils __implicitly sum over all space__ when collecting the signal, we can interpret the signal as a spatial fourier transform.

In math, each sample of the received signal is a sample of
$$
s(k_x, k_y) \;=\; \underbrace{\int}_{\text{sum over space}} \underbrace{m(x,y)}_{\text{image}} \; \underbrace{e^{-j\, 2\pi (k_x x + k_y y)}}_{\text{spatial phase}} \,dx\,dy.
$$

Let's assume you've already excited a slice and you want to encode spatial position $(x, y)$ within that slice. For simplicity, let's start with a single dimension; that is, you are imaging a line of spins (perhaps, a tube phantom filled with water and oil) which lives in the excited slice. Let's call this $M_{xy}(x)$. We will apply a gradient along $\hat x$ after these spins are excited. 

Using the rotating frame (that is, ignoring the effect of $B_0$), a spin at position $x$ feels
$$
B_z(x) = G_x\, x
$$
and its transverse magnetization therefore precesses at
$$
\omega(x) = \gamma G_x x.
$$

Clearly, spins at different $x$ pick up phase at different rates. You can imagine these as a bunch of little spinning arrows along the line we're imaging. Let's do a simple example to demonstrate: in the rotating frame, the only $z$-field is $G_x x$, so a spin sitting at $x = 0$ should *not* precess, and a spin at $x > 0$ _should_. Let's tip _only_ two spins by 90 degrees, turn on $G_x$, and confirm this.

In [ ]:
dt = 1e-6
n_steps = 4000
Gx = 0.05  # G/cm

# TODO: define two spins, both with positions on the x axis, after a 90-degree excitation.
# HINT: use help(Spin) for a refresher on how to define a spin.
spin_iso = # ____
spin_off = # ____
M_iso = np.zeros((3, n_steps))
M_off = np.zeros((3, n_steps))

for step in range(n_steps):
    M_iso[:, step] = spin_iso.magnetization
    M_off[:, step] = spin_off.magnetization
    # TODO: update both spins. 
    # HINT 1: define B as a vector. recall, B is zero in the rotating frame.
    # HINT 2: define G as a vector. recall, G is a vector of the gradient strengths.
    # HINT 3: spin.update(B, dt, G=G) (this is new!) updates the spin using the gradient.
    B = # ____
    G = # ____
    spin_iso.update(# ____)
    spin_off.update(# ____)

t = np.arange(n_steps) * dt
plot_vector_components(
    [M_iso, M_off], t,
    names=(f'isocenter ($x={spin_iso.position[0]:.1f}$ cm)', 
            f'off-center ($x={spin_off.position[0]:.1f}$ cm)'),
    title='Isocenter vs off-center spin under $G_x$',
)


You should observe one spin sitting still and the other spinning. ___This is frequency encoding___: spins at different positions precess at different frequency. Ultimately, we can abstract away most of the physics to say that our *received signal* is the vector sum of an *ensemble* of these arrows, one at each position $x$.

Let's simulate a one-dimensional object and watch this. First, use the provided function `two_block_object` to define a $1\text{D}$ phantom. This is, in other words, a sequence of discrete spins with varying amplitude. You are also free to examine this function's implementation and modify it, or generate another $1\text{D}$ phantom of your own!


In [ ]:
N = 32
fov = 2.0  # cm
dt = 1e-3
x = (np.arange(N) - N // 2) * (fov / N)

# TODO: make a 1D object M0(x). you can use two_block_object, or your own.
M0 = # ____

# TODO: spawn a Spin at each x, with that spin's M0
# HINT: make_line_spins does this with the output of two_block_object
#       but you can also write a for loop like you did in lab 3
spins = # ____

plot_1D_phantom(x, M0, title=r'Your $1\text{D}$ Phantom')


The receive coils record a single complex number: the sum of every spin's transverse magnetization. Write a function now which returns this received signal. We will use it throughout the rest of the lab.

In [ ]:
# TODO: Implement this function.
# The receive coil records the sum of every spin's transverse magnetization.
def receive_signal(spins):
    # HINT: spin.mxy is the complex number representing a single spin's transverse magnetization
    pass


You already learned the nitty-gritty of RF excitation in Lab 3. Let's abstract all that away for now; instead, write a function `instant_90` which tips (perhaps, a better word is "teleports") every spin into the transverse plane.

In [ ]:
# TODO: Implement this function.
# Instant 90-degree tip: put each spin's M0 into the transverse plane along +x.
def instant_90(spins):
    # HINT: spin.tip_90() does this for *one* spin
    pass


def reset_and_tip(spins):
    """ fully relaxes the spins, then uses instant_90. helpful for the rest of the lab. """
    for spin in spins:
        spin.reset()
    instant_90(spins)


Next we'll tip the whole object 90 degrees, play $G_x$ for a while, and look at the little arrows along $x$. I'll print the net signal with `receive_signal`.

Run the next cell to play an animation of the spins dephasing under a constant $G_x$. Hit play and _watch $|s|$ as the arrows fan out_. Try experimenting with different `Gx` too.

In [ ]:
# change Gx and re-run this cell
Gx = 0.05  # G/cm
interactive_dephasing(x, M0, Gx=Gx, duration=0.02, dt=dt, tip_fn=instant_90)


You should have observed that when you crank up $|G_x|$ and re-run the animation, or when you wait longer with the same $|G_x|$, the net signal $|s|$ shrinks. Why? Is each spin's magnitude $|M_{xy}|$ dying due to $T_2$, or is something else going on?


<span style="color:cyan">
Your answer here.
</span>

Based on your answer above, we can say that the phase each spin has accumulated is some function of 
+ time passed,
+ gradient waveforms,
+ and position.

Recall from class that phase accumulation is written as multiplication by a complex exponential 
$$ M_{xy}(x) e^{i \phi(t, G, x)} = M_{xy}(x) e^{-i\, 2\pi k_x(t)\, x} \qquad \text{(since phase is linear in position)} $$
where $k_x(t)$ is a function of both the __gradient waveform__ and the __total time the spins have experienced that waveform__, 

$$ k_x(t) = \frac{\gamma}{2\pi}\int_0^t G_x(\tau)\,d\tau. $$

Way back in lab 1 you wrote something like $k = (\gamma/2\pi)\, G\, T$ after hearing a very high level description of MRI spatial encoding. Verify for yourself that the equation we just wrote means the _same thing_! Next, let's simulate the collection of that signal during a constant $G_x$ readout, and keep track of how far we've "walked" in k-space.

In [ ]:
reset_and_tip(spins)

# not imaging parameters — just watch |s| die as we walk away from k = 0
n_read = 40
Gx = 0.05  # G/cm
B = [0.0, 0.0, 0.0]

# TODO: apply G for n_read steps and record the signal each step
# HINT: sample *before* you update, so the first sample is still at k = 0
s = np.zeros(n_read, dtype=complex)
for step in range(n_read):
    s[step] = # ____
    for spin in spins:
        spin.update(# ____)

t = np.arange(n_read) * dt

# TODO: write the k_x(t) coordinates for a constant Gx, starting at 0
kx = # ____

plot_signal_and_k(t, s, kx, title='Constant $G_x$ readout, starting at $k=0$')


This should agree with the observations you made above: as time goes on, the signal decays as the spins dephase. Explain this using the k-space interpretation _and_ the sum-of-vectors interpretation.

<span style="color:cyan">
Your answer here.
</span>

We have almost enough to reconstruct an image; the signal sampled at each point corresponds exactly to a sample of the object's spatial fourier transform. 

Here's one small problem: this readout only visited *positive* $k_x$. A real object has negative spatial frequencies too. The usual solution is the use of a *prewinder* — a negative gradient that shifts the "starting position" to $-k_\text{max}$ *before* we apply gradients and start the readout. Then, the received signal corresponds to samples of k space starting with $-k_\text{max}$, moving through $0$, and ending at $+k_\text{max}$.

We also need to consider the object's $FOV$ and resolution:
+ frequency resolution and image FOV are dual: choose `Gx` so that each sample moves in $k_x$ by exactly $\delta k = 1/\mathrm{FOV}$
+ frequency FOV and image resolution are dual: choose `n_read` so that $W_k = n_{\text{read}} \delta_k = 1 / \delta x = N / \text{FOV}$

Account for these (prewinder, FOV, and resolution) by filling in the cell below.


In [ ]:
reset_and_tip(spins)

n_read = N
dt_fast = 0.5*dt # you might want to use this later...

# TODO: calculate a readout gradient which steps kx by 1/fov
# HINT: at each sample, we move in k space by Δk = (γ/2π) G Δt
Gx = # ____
G_read = np.full(n_read, Gx)

# TODO: one-step prewinder that sends k to -kmax = -(N // 2) / FOV
# HINT: G_pre = -(n_read // 2) * Gx
G_pre = # ____
for spin in spins:
    spin.update([0.0, 0.0, 0.0], dt, G=[G_pre, 0.0, 0.0])

k0 = (GAMMA / (2 * np.pi)) * G_pre * dt

# TODO: play the readout and collect s
# HINT: try help(simulate_waveform) to use the provided helper function
s, k, _ = # ____

plot_signal_and_k(np.arange(n_read) * dt, s, k, title='Prewinder + readout (full $k_x$ line)')


The final step is to run an inverse FFT on the data. Note that the data we collected is centered. Hence, you can use the provided function `centered_ifft`.

In [ ]:
# TODO: reconstruct by inverse FFT
img = # ____
plot_object_and_recon(x, M0, img)


Confirm that the blue _reconstructed $1\text{D}$ image_ sits right on top of the black _real spin phantom_. We just *imaged* something with a Bloch simulator. How cool!

Suppose you double $G_x$ but keep the same number of samples $N$ and the same dwell $\Delta t$. What happens to the field of view? What happens to the spatial resolution? Re-run the two cells above with twice (or half) the `Gx` you just calculated to confirm that you are right. Note that when you do this, the x axis won't really make sense for your reconstructed image.

Finally, what happens if you double `Gx` _and_ halve `dt`? Verify your answer. In _this_ case, the x-axis _will_ make sense for your reconstructed image. Why?

<span style="color:cyan">
Your answer here.
</span>

## Part 2: Is the Recorded Signal _Really_ K-Space?

It is absolutely ___fascinating___ to me that the MRI physics ___literally does a fourier transform___ on the object we are imaging! Let's test it by taking analytically the FFT of $M_{xy}(x)$. This _should match_ the Bloch-simulated $s(k)$ at the same $k$ coordinates.


In [ ]:
# TODO: compute the FFT of the object (hint: centered_fft)
S_theory = # ____

plt.figure(figsize=(8, 4))
plt.plot(k[0], np.abs(s), 'C0o-', label='Bloch-simulated $|s(k)|$')
plt.plot(k[0], np.abs(S_theory), 'k--', label='DFT of the object')
plt.xlabel('$k_x$ (1/cm)')
plt.ylabel('magnitude')
plt.title('The signal *is* the Fourier transform')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()


Do the curves align? Is this cool? Watch the movie below that reconstructs with progressively more samples as the "scanner" traverses from $-k_\text{max}$ toward $+k_\text{max}$. Predict what the image looks like *before* we pass $k = 0$, and *right after*. Would an MRI scan that doesn't collect signal at $k=0$ give a good image? Why or why not?

<span style="color:cyan">
Your answer here.
</span>

In [ ]:
animate_running_recon(k[0], s, x, M0, fps=6)


## Part 3: Imaging a $2\text{D}$ Phantom in $1\text{D}$

Let's make a $2\text{D}$ phantom. As before, I implemented a phantom-maker for you, but you are welcome to tweak this or to make your own.


In [ ]:
N_phase = 13
N_freq = 13
fov = 2.0  # cm
dt = 1e-3

x2 = (np.arange(N_freq) - N_freq // 2) * (fov / N_freq)
y2 = (np.arange(N_phase) - N_phase // 2) * (fov / N_phase)

# TODO: a 2D object on this grid
# HINT: plus_phantom
M0_2d = # ____

# TODO: spawn a Spin at each (x, y)
# HINT: make_grid_spins
spins_2d = # ____

plot_2d_phantom(x2, y2, M0_2d, title=r'Your $2\text{D}$ Phantom')


Note that we can reuse `instant_90` to tip all the spins into the transverse plane. First, let's try playing a gradient in $x$ only. We should see very similar behavior as what we saw in part 1. We'll also observe $|s|$ with your same `receive_signal`.

Run the next cell to play an animation of the $2\text{D}$ grid dephasing under a constant $G_x$. Hit play and _watch $|s|$ as the arrows fan out_. Try experimenting with different `Gx` too. Then answer these three questions:
+ Do you notice the same signal decay as you did in part 1? Why or why not?
+ Notice with positive $G_x$, after some fixed time $T$, the spins with fixed $y$ coordinate have "twisted" by the same amount. Why does this make sense?
+ Why is this not enough to recover a $2\text{D}$ image? What do we need to eventually add?


In [ ]:
# change Gx and re-run this cell
Gx = 0.05  # G/cm
interactive_dephasing_2d(x2, y2, M0_2d, Gx=Gx, duration=0.02, dt=dt, tip_fn=instant_90)


<span style="color:cyan">
Your answer here.
</span>

Next, let's try applying the _same_ $1\text{D}$ imaging procedure. This will still work for our $2\text{D}$ phantom; but since we are encoding only in the $x$ direction, we will get a $1\text{D}$ projection of the phantom. What does this mean? In the case of a plus phantom, since there is more signal coming from the center region where the plus is taller, we expect the $1\text{D}$ projection to be 'taller' in the center; correspondingly there should be less signal from the 'arms' of the plus so the $1\text{D}$ projection is 'shorter' here; and there are no spins near the edges, so the $1\text{D}$ projection should be $0$.

In [ ]:
reset_and_tip(spins_2d)

# TODO: readout gradient that steps kx by 1/fov (same as Part 1)
# HINT: Gx = (1/fov) / ((GAMMA / (2*np.pi)) * dt)
Gx = # ____
G_read = np.full(N_freq, Gx)

# TODO: one-step prewinder that sends k to -kmax = -(N_freq // 2) / FOV
# HINT: G_pre = -(N_freq // 2) * Gx  (same as Part 1)
G_pre = # ____
for spin in spins_2d:
    spin.update([0.0, 0.0, 0.0], dt, G=[G_pre, 0.0, 0.0])

k0 = (GAMMA / (2 * np.pi)) * G_pre * dt

# TODO: play the readout and reconstruct
# HINT: simulate_waveform, then centered_ifft
s_proj, k_proj, _ = # ____
img_1d = # ____

# the 1D image is the phantom summed along y
projection = M0_2d.sum(axis=0)
plot_object_and_recon(x2, projection, img_1d, title=r'$1\text{D}$ projection of the plus')


___Challenge:___ go into `plus_phantom` and try to modify it such that the phantom above appears to be a rectangle; that is, it no longer has this stair-step appearance.

## Part 4: Imaging a $2\text{D}$ Phantom in $2\text{D}$

So far, we have been collecting samples of k-space in $1\text{D}$ k-space and thus recovering $1\text{D}$ images. Our goal is a two-dimensional image, so we need samples of a __two-dimensional k-space__. Repeated below, for your convenience, are the equations that tell us how the gradients in a sense 'traverse' k-space:

$$
s(k_x, k_y) \;=\; \sum_{x,y} m(x,y)\, e^{-j\, 2\pi (k_x x + k_y y)},
\qquad
(k_x, k_y) = \frac{\gamma}{2\pi}\int (G_x, G_y)\,dt.
$$

The cell below helps us visualize this. Choose some strengths for $G_x$ and $G_y$. The plot on the left tells us the k space coordinate at each discrete time step; in other words, the location of the sample of k-space `receive_signal` would give for that time. The plot on the right shows the spins' dephasing; confirm that this looks right for the special cases $G_x=0$ or $G_y=0$.

In [ ]:
n_steps = 16
G = np.zeros((2, n_steps))
G[0, :] = 0.05 # TODO: choose an x gradient
G[1, :] = 0.05 # TODO: choose a y gradient
animate_2d_encoding(x2, y2, M0_2d, G, dt, tip_fn=instant_90)


With some constraints, we have almost complete control over the gradients, so we have _complete control over where we are sampling in k-space_. 

The challenge is to sample at points that 'make sense.' What I mean by this, is that the 'default' implementations of IFFT and FFT only work on a _cartesian grid_. This means that we want to traverse and sample k-space along a cartesian grid. We usually do this one "line" at a time, with the following strategy:

1. blip $G_y$ to pick a $k_y$ (*phase encode*)
2. prewind $k_x$ (same trick as Part 1)
3. play $G_x$ and listen (*frequency encode*)
4. wait for relaxation, excite again, repeat with a different $k_y$

We call each repetition of these steps a 'TR.' In the next cell, your task is to design a single TR. Then you'll use the same visualization to confirm you got it right.

In [ ]:
# TODO: readout gradient that steps kx by 1/fov each dwell
# HINT: same as part 1!
Gx = # ____

# TODO: pick one ky and the Gy blip that lands there in one dwell
# HINT: determine the spacing between ky lines, then pick one and solve for Gy
# HINT: this should be very similar to your readout gradient calculation!
dky = 1.0 / fov
Gy = # ____

# TODO: one-step prewinder that sends k to -kmax = -(N_freq // 2) / FOV
# HINT: G_pre = -(N_freq // 2) * Gx  (same as Part 1)
G_pre = # ____

# one TR: phase-encode blip, prewinder, then readout
G_tr = np.zeros((2, 2 + N_freq))
G_tr[1, 0] = Gy
G_tr[0, 1] = G_pre
G_tr[0, 2:] = Gx

animate_2d_encoding(x2, y2, M0_2d, G_tr, dt, tip_fn=instant_90)


That's *one* line of k-space. A $2\text{D}$ image is many lines — one $k_y$ per TR. Your final task is to write the list of $k_y$ locations.

In [ ]:
# TODO: list of ky locations, one per phase-encode line
# HINT: np.arange(N_phase) gives you a list [0, 1, 2, ..., N_phase-1]
# HINT: but you want [-(N_phase//2)*dky, ..., 0, ..., +(N_phase//2)*dky]
ky_list = # ____

kspace = np.zeros((N_phase, N_freq), dtype=complex)
G_read = np.full(N_freq, Gx)

for i, ky in enumerate(ky_list):
    # TODO: new TR — reset, tip, blip Gy, prewind kx, read
    # HINT 1: first, reset and re-excite the spins
    # HINT 2: compute Gy for *this* TR using ky_list
    # HINT 3: apply the prewinder / Gy blips
    # HINT 4: simulate the readout with simulate_waveform!
    reset_and_tip(# ____)
    Gy = # ____
    for spin in spins_2d:
        spin.update(# ____)
        spin.update(# ____)
    s_line, _, _ = # ____
    kspace[i, :] = s_line


In [ ]:
# TODO: 2D inverse FFT of centered k-space
# HINT: centered_ifft2
img2 = # ____

plot_kspace_and_image(
    kspace, img2,
    extent=[x2[0], x2[-1], y2[0], y2[-1]],
    titles=('|k-space|', 'reconstruction'),
)


You should see the plus! Question: Why re-excite (a new TR) for every phase-encode line? Can you just keep the same transverse magnetization and just blip $G_y$ again? Why or why not?

<span style="color:cyan">
Your answer here.
</span>

If you swapped the roles of $G_x$ and $G_y$ (phase encode along $x$, frequency encode along $y$), without changing anything else, what would happen to the reconstructed image? Would the reconstructed plus phantom look different?

<span style="color:cyan">
Your answer here.
</span>

## Part 5: Is the Recorded Signal _Really_ K-Space? (Encore)

In part 2, we checked this in one dimension: the Bloch-simulated $s(k)$ sat right on top of the raw FFT of the object. Let's prove the same claim in two dimensions.


In [ ]:
# TODO: compute the 2D FFT of the object
# HINT: centered_fft2
S_theory_2d = # ____

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(np.abs(kspace), cmap='gray', origin='lower')
axes[0].set_title(r'Bloch-simulated $|s(k_x, k_y)|$')
axes[0].set_xlabel('$k_x$ index')
axes[0].set_ylabel('$k_y$ index')
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(np.abs(S_theory_2d), cmap='gray', origin='lower')
axes[1].set_title('DFT of the object')
axes[1].set_xlabel('$k_x$ index')
axes[1].set_ylabel('$k_y$ index')
fig.colorbar(im1, ax=axes[1], fraction=0.046)
fig.tight_layout()
plt.show()


Do the two images match? Is this still cool? Watch a reconstruction below that only gets to use the phase-encode lines collected *so far*, as the "scanner" walks from most-negative $k_y$ toward most-positive $k_y$. Predict what the image looks like *before* we acquire the $k_y = 0$ line, and *right after*. Would an MRI scan that skips the center of k-space give a good image? Why or why not?


<span style="color:cyan">
Your answer here.
</span>

In [ ]:
animate_running_recon_2d(
    kspace, M0_2d,
    extent=[x2[0], x2[-1], y2[0], y2[-1]],
    fps=4,
)


Hopefully this reminds you of the following GIF, which I've used over and over to advertise this class:

![top_to_bottom_cartesian.gif](top_to_bottom_cartesian.gif)

Of course, your simulation is using a lot fewer pixels than you'd find in images reconstructed by a real MRI scan.

Nice! You made it through the lab where MRI actually starts to look like *imaging*. You encoded position with frequency and phase using gradients, filled k-space in 1D and 2D, and used an inverse FFT to reconstruct some simple phantoms. Later on we might do fancier things with how we sample k-space. If nothing else, I hope you walk away from this lab __believing that MRI physics performs a fourier transform__.

___this lab and accompanying visualization written by Ryan Black for the MRI decal, fall 2026___